# Mouse decidualization: in vitro DEGs vs natural pregnancy, artificial decidualization and X-Mens

**Reference (in vitro):** Wang et al. 2020 IVD - mouse endometrial stromal cells, E2 + P4 for 4 days vs untreated (Table S6).

Compared against:

| Dataset | Comparison | Source |
|---|---|---|
| NPD | day 8 implantation sites vs inter-implantation sites (whole uterus) | Wang et al. 2020, Table S2 |
| AD | day 8 oil-stimulated vs unstimulated horn (whole uterus) | Wang et al. 2020, Table S3 |
| X-Mens | `dec` vs `eFib`, pseudobulk, paired within animal | this study |

Wang et al. called DEGs with **fold change >= 2 and FDR <= 0.01** (TopHat/Cufflinks, mm9, n = 3).
Their tables contain only DEGs, so for NPD and AD a gene missing from the table is "not a DEG"
(not significant or not tested). The same cutoffs are applied to our data by default.

In X-Mens, `timepoint` identifies the biological sample (one animal per timepoint); the 10x lanes
in `sample` are summed into their animal, so lanes are never treated as replicates. The animals
(`INCLUDE_TP`) and any excluded cell subtypes (`EXCLUDE_SUBTYPES`) are set in section 3.

**Outputs** (in `OUT_DIR`):
* `XMens_dec_vs_eFib_DEGs.csv` (and `_all_genes.csv`) - X-Mens DEGs
* `IVD_DEGs_vs_invivo.csv` - every IVD DEG with fold change, FDR and direction in NPD, AD and our data
* `IVD_DEGs_vs_invivo_stacked.svg` (and `_stacked_counts.csv`) - % of IVD DEGs shared with each in vivo dataset

## 1. Libraries

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import scipy.sparse as sp
import scanpy as sc
import matplotlib.pyplot as plt

from pydeseq2.dds import DeseqDataSet
from pydeseq2.ds import DeseqStats
from pydeseq2.default_inference import DefaultInference

plt.rcParams["svg.fonttype"] = "none"      # keep text editable in the SVG
pd.set_option("display.width", 200)

## 2. Datasets

In [ ]:
adata = sc.read_h5ad("/directoryto/data/20261111_mouse_only_eFib_dec_final.h5ad")

BULK_DIR = Path("/directoryto/BulkRnaSeq")
WANG_DIR = BULK_DIR / "Wang2020"

print(f"Our snRNA-seq: {adata.n_obs:,} cells x {adata.n_vars:,} genes")
print(adata.obs["cell_type"].value_counts().to_string())


def read_wang(table_no):
    """Read a Wang et al. 2020 supplementary DEG table (title in row 1, header in row 3)."""
    path = sorted(WANG_DIR.glob(f"*Supplementary_Table_{table_no}.xlsx"))
    if not path:
        raise FileNotFoundError(f"No *Supplementary_Table_{table_no}.xlsx in {WANG_DIR}")
    title = pd.read_excel(path[0], header=None, nrows=1).iloc[0, 0]
    d = pd.read_excel(path[0], header=2).iloc[:, :9]
    d.columns = ["gene_symbol", "gene_id", "transcript_id", "mean_treated", "mean_control",
                 "fold_change", "pvalue", "FDR", "description"]
    d = d.dropna(subset=["gene_id"])
    d["gene_id"] = d["gene_id"].astype(str).str.split(".").str[0]
    d["direction"] = np.where(d["fold_change"] > 1, "up", "down")
    d["control_zero"] = d["mean_control"] == 0          # fold change is a placeholder for these
    d["log2FC"] = np.log2(d["fold_change"].where(~d["control_zero"]))
    print(f"Table S{table_no}: {len(d)} DEGs (up {(d.direction == 'up').sum()}, "
          f"down {(d.direction == 'down').sum()})  - {title}")
    return d.set_index("gene_id")

WANG = {
    "IVD": read_wang(6),     # in vitro decidualization (reference)
    "NPD": read_wang(2),     # natural pregnancy
    "AD":  read_wang(3),     # artificial decidualization
}

## 3. Variables

In [ ]:
# X-Mens animals used (paired dec vs eFib within each animal)
INCLUDE_TP = ["3.5dpi", "4.5dpi_menstrual"]

# Cell subtypes removed before pseudobulking (empty list = keep all)
SUBTYPE_COL = "cell_subtype"
EXCLUDE_SUBTYPES = ["decAp"]

MIN_CELLS = 20                       # min cells per animal x group pseudobulk

tp = adata.obs["timepoint"].astype(str)
st = adata.obs[SUBTYPE_COL].astype(str)
unknown = set(INCLUDE_TP) - set(tp.unique())
assert not unknown, f"Not found in obs['timepoint']: {unknown}. Available: {sorted(tp.unique())}"
unknown = set(EXCLUDE_SUBTYPES) - set(st.unique())
assert not unknown, f"Not found in obs['{SUBTYPE_COL}']: {unknown}. Available: {sorted(st.unique())}"
assert len(INCLUDE_TP) >= 2, "Need at least 2 animals for a paired comparison"

adata_x = adata[tp.isin(INCLUDE_TP).values & ~st.isin(EXCLUDE_SUBTYPES).values]

print(f"Animals: {INCLUDE_TP}")
print(f"Excluded subtypes: {EXCLUDE_SUBTYPES if EXCLUDE_SUBTYPES else 'none'}  ({adata_x.n_obs:,} cells kept)")
print("\nSubtypes kept (cell type x subtype):")
print(pd.crosstab(adata_x.obs["cell_type"].astype(str), adata_x.obs[SUBTYPE_COL].astype(str)).to_string())
n_cells = pd.crosstab(adata_x.obs["timepoint"].astype(str), adata_x.obs["cell_type"].astype(str))
print("\nCells kept (animal x cell type):")
print(n_cells.to_string())
low = n_cells[["dec", "eFib"]].min(axis=1) < MIN_CELLS
assert not low.any(), f"Fewer than {MIN_CELLS} dec or eFib cells in: {list(n_cells.index[low])}"

DATASETS = {
    "XMens_dec_vs_eFib": dict(
        label="X-Mens dec vs eFib",
        adata=adata_x,
        label_col="cell_type",
        groups={"dec": "dec", "eFib": "nonDec"},
        sample_key="timepoint",                        # animal ("sample" = 10x lanes, summed)
        counts=("layer", "raw_counts"),
        gene_id="gene_ids",
        symbol="index",
    ),
}

TEST, REF = "dec", "nonDec"          # log2FC > 0 = higher in decidual cells

# DEG cutoffs - Wang et al. 2020 used FC >= 2 and FDR <= 0.01
FDR, FC, MIN_CPM = 0.01, 2.0, 3.0

# Bars in the plot: in vivo datasets compared with the IVD DEGs
COMPARE = {
    "NPD": "Natural pregnancy\n(NPD)",
    "AD": "Artificial\ndecidualization (AD)",
    "XMens_dec_vs_eFib": "X-Mens\ndec vs eFib",
}

OUT_DIR = BULK_DIR / "SharedDEGs" / "Mouse_InVivo_vs_InVitro"
OUT_DIR.mkdir(parents=True, exist_ok=True)

MARKERS = ["Prl8a2", "Alpl", "Bmp2", "Wnt4", "Hand2", "Ptgs2", "Hoxa10", "Foxo1", "Pgr", "Esr1"]

## 4. Build pseudobulk counts (sum of raw counts per animal x cell type)

In [ ]:
def build_pseudobulk(cfg):
    a = cfg["adata"]
    group = a.obs[cfg["label_col"]].astype(str).map(cfg["groups"])
    keep = group.notna().values
    obs = pd.DataFrame({"donor": a.obs[cfg["sample_key"]].astype(str).values[keep],
                        "condition": group.values[keep]})

    kind, key = cfg["counts"]
    if kind == "raw":
        X, var = a.raw.X, a.raw.var
    else:
        X, var = a.layers[key], a.var
    X = sp.csr_matrix(X)[keep]

    def from_var(spec):
        s = pd.Series(var.index.astype(str), index=var.index) if spec == "index" else \
            (var[spec] if spec in var.columns else a.var[spec].reindex(var.index))
        return s.astype(str).values
    gene_ids = pd.Index(from_var(cfg["gene_id"])).str.split(".").str[0]
    symbols = from_var(cfg["symbol"])

    print(pd.crosstab(obs["donor"], obs["condition"]).T.to_string(), "\n")

    obs["pb_id"] = obs["donor"] + "__" + obs["condition"]
    codes, pb_ids = pd.factorize(obs["pb_id"])
    onehot = sp.csr_matrix((np.ones(len(codes)), (codes, np.arange(len(codes)))),
                           shape=(len(pb_ids), len(codes)))
    pb = pd.DataFrame((onehot @ X).toarray(), index=pb_ids, columns=gene_ids)
    pb = pb.T.groupby(level=0).sum().T                       # merge duplicated gene IDs

    meta = obs.groupby("pb_id").agg(donor=("donor", "first"), condition=("condition", "first"),
                                    n_cells=("donor", "size")).loc[pb.index]
    small = meta.n_cells < MIN_CELLS
    meta, pb = meta[~small], pb[~small]
    print(f"Dropped {small.sum()} pseudobulks with < {MIN_CELLS} cells")

    both = meta.groupby("donor")["condition"].nunique()
    paired = (both == 2).sum() >= 2
    if paired:
        meta = meta[meta.donor.isin(both[both == 2].index)]
        pb = pb.loc[meta.index]
    design = "~donor + condition" if paired else "~condition"

    sym_map = pd.Series(symbols, index=gene_ids)
    sym_map = sym_map[~sym_map.index.duplicated()]
    print(f"{len(meta)} pseudobulks from {meta.donor.nunique()} animals; design: {design}")
    print(meta.groupby("condition")["n_cells"].agg(["count", "sum", "median"]).to_string())
    return dict(pb=pb, meta=meta, design=design, sym_map=sym_map)


PB = {}
for name, cfg in DATASETS.items():
    print(f"\n{'=' * 30} {cfg['label']} {'=' * 30}")
    PB[name] = build_pseudobulk(cfg)

## 5. PyDESeq2

In [ ]:
def run_deseq(pbd):
    pb, meta = pbd["pb"], pbd["meta"].copy()
    counts = pb.round().astype(int)
    counts = counts.loc[:, (counts >= 10).sum(axis=0) >= 2]
    meta["condition"] = pd.Categorical(meta["condition"], categories=[REF, TEST])
    meta["donor"] = meta["donor"].astype("category")

    inference = DefaultInference(n_cpus=4)
    dds = DeseqDataSet(counts=counts, metadata=meta, design=pbd["design"],
                       refit_cooks=True, inference=inference, quiet=True)
    dds.deseq2()
    ds = DeseqStats(dds, contrast=["condition", TEST, REF], inference=inference, quiet=True)
    ds.summary()
    res = ds.results_df.copy()
    ds.lfc_shrink(coeff=f"condition[T.{TEST}]")
    res["log2FoldChange_shrunk"] = ds.results_df["log2FoldChange"]

    cpm = pb.div(pb.sum(axis=1), axis=0) * 1e6
    for lvl in (REF, TEST):
        res[f"meanCPM_{lvl}"] = cpm.loc[meta.index[meta["condition"] == lvl], res.index].mean()
    expressed = res[[f"meanCPM_{REF}", f"meanCPM_{TEST}"]].max(axis=1) > MIN_CPM

    sig = (res["padj"] <= FDR) & expressed
    lfc_cut = np.log2(FC)
    res["direction"] = "ns"
    res.loc[sig & (res["log2FoldChange"] >= lfc_cut), "direction"] = "up"
    res.loc[sig & (res["log2FoldChange"] <= -lfc_cut), "direction"] = "down"
    res.insert(0, "gene_symbol", pbd["sym_map"].reindex(res.index).values)
    res.index.name = "gene_id"
    return res.sort_values("padj")


RES = {}
for name, cfg in DATASETS.items():
    print(f"\n{'=' * 30} {cfg['label']} {'=' * 30}")
    res = run_deseq(PB[name])
    RES[name] = res
    res.to_csv(OUT_DIR / f"{name}_all_genes.csv")
    res[res.direction != "ns"].to_csv(OUT_DIR / f"{name}_DEGs.csv")
    print(f"{TEST} vs {REF}: up {(res.direction == 'up').sum()}, down {(res.direction == 'down').sum()} "
          f"(padj <= {FDR}, |FC| >= {FC}, CPM > {MIN_CPM})")
    print(res[res.gene_symbol.isin(MARKERS)][["gene_symbol", "log2FoldChange", "padj", "direction"]].to_string())

## 6. Compare with the in vitro DEGs (Wang et al. 2020 IVD)

Genes are matched on Ensembl ID. Wang et al. used an older annotation (mm9), so IDs not found
in our data are matched by gene symbol instead.
For NPD and AD, IVD DEGs absent from their tables count as "not a DEG".

In [ ]:
ivd = WANG["IVD"]
table = ivd[["gene_symbol", "direction", "fold_change", "log2FC", "pvalue", "FDR", "control_zero"]] \
          .rename(columns={"direction": "IVD_direction", "fold_change": "IVD_fold_change",
                           "log2FC": "IVD_log2FC", "pvalue": "IVD_pvalue", "FDR": "IVD_FDR",
                           "control_zero": "IVD_control_zero"})

# Wang et al. in vivo models (same study, same IDs)
for m in ("NPD", "AD"):
    w = WANG[m].reindex(table.index)
    table[f"{m}_direction"] = w["direction"].fillna("ns")
    table[f"{m}_fold_change"] = w["fold_change"]
    table[f"{m}_pvalue"] = w["pvalue"]
    table[f"{m}_FDR"] = w["FDR"]

# Our data: match by Ensembl ID, then by symbol for the rest
for name, cfg in DATASETS.items():
    res = RES[name]
    by_id = res.reindex(table.index)
    sym_res = res.reset_index().drop_duplicates("gene_symbol").set_index("gene_symbol")
    missing = by_id["direction"].isna()
    by_sym = sym_res.reindex(table.loc[missing, "gene_symbol"].values)
    by_sym.index = table.index[missing]
    r = by_id.copy()
    r.loc[missing, by_sym.columns.intersection(r.columns)] = by_sym[by_sym.columns.intersection(r.columns)]
    print(f"{cfg['label']}: {(~missing).sum()} IVD DEGs matched by Ensembl ID, "
          f"{by_sym['direction'].notna().sum()} by symbol, {by_sym['direction'].isna().sum()} not in data")

    table[f"{name}_direction"] = r["direction"].fillna("not tested")
    table[f"{name}_log2FC"] = r["log2FoldChange"].astype(float)
    table[f"{name}_pvalue"] = r["pvalue"].astype(float)
    table[f"{name}_padj"] = r["padj"].astype(float)

table.index.name = "gene_id"
table = table.sort_values(["IVD_direction", "IVD_FDR"], ascending=[False, True])
table.to_csv(OUT_DIR / "IVD_DEGs_vs_invivo.csv")
print(f"\nIVD DEGs: {len(table)} (up {(table.IVD_direction == 'up').sum()}, "
      f"down {(table.IVD_direction == 'down').sum()})")

# Markers across all datasets
mk = pd.concat([WANG[m][["gene_symbol", "direction"]].rename(columns={"direction": m}).set_index("gene_symbol")
                for m in ("IVD", "NPD", "AD")], axis=1)
for name in DATASETS:
    mk[name] = RES[name].drop_duplicates("gene_symbol").set_index("gene_symbol")["direction"]
print("\nMarker genes (missing = not a DEG / not in data):")
print(mk.reindex(MARKERS).fillna("-").to_string())
print(f"\nSaved {OUT_DIR / 'IVD_DEGs_vs_invivo.csv'}")

## 7. Plot Results
% of IVD DEGs, classified by their result in each in vivo dataset.

In [ ]:
CATS = {
    "Shared, up in both":         "#c0392b",   # red
    "Shared, down in both":       "#f1948a",   # light red
    "Shared, opposite direction": "#4d4d4d",   # dark gray
    "In vitro DEG only":          "#d0d0d0",   # light gray
}

bars = {}
for key, label in COMPARE.items():
    b = table["IVD_direction"]
    s = table[f"{key}_direction"].replace("not tested", "ns")
    bars[label] = {
        "Shared, up in both":         ((b == "up") & (s == "up")).sum(),
        "Shared, down in both":       ((b == "down") & (s == "down")).sum(),
        "Shared, opposite direction": ((s != "ns") & (b != s)).sum(),
        "In vitro DEG only":          (s == "ns").sum(),
    }
bar_counts = pd.DataFrame(bars).T[list(CATS)]
bar_pct = bar_counts.div(bar_counts.sum(axis=1), axis=0) * 100

flat = lambda df: df.rename(index=lambda s: s.replace("\n", " "))
print(flat(bar_counts).assign(total=bar_counts.sum(axis=1).values).to_string(), "\n")
print(flat(bar_pct).round(1).to_string())
flat(bar_counts).assign(total=bar_counts.sum(axis=1).values).to_csv(OUT_DIR / "IVD_DEGs_vs_invivo_stacked_counts.csv")

fig, ax = plt.subplots(figsize=(1.4 * len(bar_pct) + 3, 4.8))
x = np.arange(len(bar_pct))
bottom = np.zeros(len(bar_pct))
for cat, color in CATS.items():
    v = bar_pct[cat].values
    ax.bar(x, v, bottom=bottom, color=color, width=0.65, label=cat)
    for xi, vi, b0 in zip(x, v, bottom):
        if vi >= 3:
            ax.text(xi, b0 + vi / 2, f"{vi:.0f}%", ha="center", va="center", fontsize=8,
                    color="white" if color in ("#c0392b", "#4d4d4d") else "black")
    bottom += v
ax.set_xticks(x)
ax.set_xticklabels(bar_pct.index, fontsize=8)
ax.set_ylabel(f"% of in vitro (IVD) DEGs (n = {len(table)})")
ax.set_ylim(0, 100)
ax.spines[["top", "right"]].set_visible(False)
ax.legend(frameon=False, fontsize=8, bbox_to_anchor=(1.02, 1), loc="upper left")
fig.tight_layout()
fig.savefig(OUT_DIR / "IVD_DEGs_vs_invivo_stacked.svg", bbox_inches="tight")
plt.show()
print(f"Saved {OUT_DIR / 'IVD_DEGs_vs_invivo_stacked.svg'}")